In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("1. 데이터를 로드하는 중입니다...")
train_base = pd.read_csv('train_base_4800.csv')  # 2-2단계에서 정제된 4,800건
total_all = pd.read_csv('final_merged_reviews_all.csv')  # 전체 15만 건 통합본
test_set = pd.read_csv('test_set_clean_200.csv')  # 숨겨둔 200건 시험지

# 데이터 오염 방지 (이미 사용한 5,000건은 대량 라벨링 대상에서 제외)
labeled_texts = set(train_base['text'].tolist() + test_set['text'].tolist())
unlabeled_df = total_all[~total_all['text'].isin(labeled_texts)].copy()

# 2. 1차 인공지능 모델 학습 시작
print("\n2. 1차 모델 학습 시작...")
vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_base['text'])
y_train = train_base['is_spoiler']

# 불균형 데이터 처리를 위해 가중치(class_weight='balanced')를 적용합니다.
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# 3. 남은 대용량 데이터의 스포일러 확률 및 확신도 계산
print("\n3. 남은 대용량 데이터의 스포일러 확률 계산 중...")
X_unlabeled = vectorizer.transform(unlabeled_df['text'])
probs = model.predict_proba(X_unlabeled)

unlabeled_df['spoiler_prob'] = probs[:, 1]
# 확신도 계산: 50% 확률에서 멀어질수록 확실한 데이터로 판단합니다.
unlabeled_df['confidence'] = (unlabeled_df['spoiler_prob'] - 0.5).abs()

# 4. 확신도 기준 정렬 후 필요한 만큼만 채워서 10만 건 맞추기
print("\n4. 가장 확실한 데이터만 골라내어 통합 10만 건을 만드는 중...")
needed_count = 100000 - train_base.shape[0]  # 10만 건을 채우기 위해 필요한 개수 (95,200건)

# 확신도 높은 순으로 정렬 후 상위 95,200건 추출
pseudo_labeled_df = unlabeled_df.sort_values(by='confidence', ascending=False).head(needed_count).copy()
pseudo_labeled_df['is_spoiler'] = (pseudo_labeled_df['spoiler_prob'] >= 0.5).astype(int)

# 5. 기본 데이터(4,800) + 확실한 데이터(95,200) 병합 = 정확히 10만 건
final_train_df = pd.concat([
    train_base[['movie_title', 'text', 'is_spoiler']], 
    pseudo_labeled_df[['movie_title', 'text', 'is_spoiler']]
], ignore_index=True)

print(f"\n✨ [성공] 최종 학습용 10만 건 데이터셋 구축 완료: {final_train_df.shape[0]}건")
print(f"   - 정상 리뷰(0) 개수: {(final_train_df['is_spoiler'] == 0).sum()}건")
print(f"   - 스포일러 리뷰(1) 개수: {(final_train_df['is_spoiler'] == 1).sum()}건")

# 마스터 파일 저장
output_file = 'final_100k_train_set.csv'
final_train_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✅ 최종 학습용 파일이 '{output_file}'로 저장되었습니다.")

1. 데이터를 로드하는 중입니다...

2. 1차 모델 학습 시작...

3. 남은 대용량 데이터의 스포일러 확률 계산 중...

4. 가장 확실한 데이터만 골라내어 통합 10만 건을 만드는 중...

✨ [성공] 최종 학습용 10만 건 데이터셋 구축 완료: 100000건
   - 정상 리뷰(0) 개수: 90282건
   - 스포일러 리뷰(1) 개수: 9718건
✅ 최종 학습용 파일이 'final_100k_train_set.csv'로 저장되었습니다.
